# TikTok Viral Prediction System
## A Machine Learning Approach to Predicting Content Virality

**Objective:** Build an intelligent system that predicts whether a TikTok video/song/trend will go viral and identifies key factors driving success.

**Master Prompt:** You are a senior data scientist and machine learning engineer tasked with designing and building a TikTok Viral Prediction System that leverages insights from multiple datasets including:
- TikTok Trending Tracks (audio features + Spotify comparison potential)
- TikTok Viral Trends 2025 (trend summaries like challenges, sounds, songs)
- TikTok Trending Data (weekly popularity trends and engagement signals)

In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✅ All required libraries imported successfully")

---

## 📊 Section 1: Load and Explore Datasets

Load TikTok Trending Tracks, Viral Trends 2025, and Trending Data datasets. Perform exploratory data analysis to understand structure, identify key features (engagement metrics, audio features, trend types), check for missing values, and visualize distributions of viral vs non-viral content.

In [ ]:
# 2. Load Datasets
import os

# Define data paths
data_path = '../data/raw/'

# List available files
print("📁 Available datasets in /data/raw/:")
datasets = {}
if os.path.exists(data_path):
    files = os.listdir(data_path)
    for f in files:
        if f.endswith('.csv'):
            file_path = os.path.join(data_path, f)
            try:
                datasets[f.replace('.csv', '')] = pd.read_csv(file_path)
                print(f"   ✅ Loaded: {f}")
            except Exception as e:
                print(f"   ⚠️  Could not load {f}: {e}")
else:
    print(f"   ⚠️  Data path not found: {data_path}")

# 3. Exploratory Data Analysis (EDA) - Dataset Overview
print("\n" + "=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

for dataset_name, df in datasets.items():
    print(f"\n📊 Dataset: {dataset_name}")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Sample data:")
    print(df.head(3))
    print(f"\n   Data types: {df.dtypes.to_dict()}")
    print(f"   Missing values: {df.isnull().sum().sum()} total")

# 4. Identify Key Features and Data Characteristics
print("\n" + "=" * 80)
print("KEY FEATURES ANALYSIS")
print("=" * 80)

for dataset_name, df in datasets.items():
    print(f"\n📌 {dataset_name}:")
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    print(f"   Numeric features ({len(numeric_cols)}): {numeric_cols}")
    print(f"   Categorical features ({len(categorical_cols)}): {categorical_cols}")
    
    # Identify potential engagement metrics
    engagement_keywords = ['view', 'like', 'share', 'comment', 'engagement', 'followers', 'count']
    engagement_cols = [col for col in numeric_cols if any(kw in col.lower() for kw in engagement_keywords)]
    if engagement_cols:
        print(f"   Potential engagement metrics: {engagement_cols}")
    
    # Identify potential audio features
    audio_keywords = ['energy', 'danceability', 'tempo', 'valence', 'acoustic', 'loud', 'speech']
    audio_cols = [col for col in numeric_cols if any(kw in col.lower() for kw in audio_keywords)]
    if audio_cols:
        print(f"   Potential audio features: {audio_cols}")

---

## ⚙️ Section 2: Feature Engineering

Create derived features including engagement velocity (growth rate over time), audio-feature similarity scores to known viral songs, trend category encoding (challenge/sound/meme), recency factors, and virality score labels.

In [ ]:
# 5. Feature Engineering - Create Derived Features

def create_viral_label(df, engagement_cols, threshold_percentile=75):
    """Create virality label based on engagement metrics."""
    if not engagement_cols:
        return None
    engagement_score = df[engagement_cols].fillna(0).sum(axis=1)
    threshold = engagement_score.quantile(threshold_percentile / 100)
    viral_label = (engagement_score > threshold).astype(int)
    return viral_label, engagement_score

def engineer_features(df, dataset_name):
    """Create engineered features from raw data"""
    df_engineered = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 1. Engagement features
    engagement_keywords = ['view', 'like', 'share', 'comment', 'engagement', 'followers']
    engagement_cols = [col for col in numeric_cols if any(kw in col.lower() for kw in engagement_keywords)]
    
    if engagement_cols:
        print(f"   Creating engagement features from: {engagement_cols}")
        df_engineered['engagement_total'] = df[engagement_cols].fillna(0).sum(axis=1)
        df_engineered['engagement_mean'] = df[engagement_cols].fillna(0).mean(axis=1)
        df_engineered['engagement_std'] = df[engagement_cols].fillna(0).std(axis=1)
    
    # 2. Audio features
    audio_keywords = ['energy', 'danceability', 'tempo', 'valence', 'acoustic', 'loud', 'speech']
    audio_cols = [col for col in numeric_cols if any(kw in col.lower() for kw in audio_keywords)]
    
    if audio_cols:
        print(f"   Creating audio features from: {audio_cols}")
        for col in audio_cols:
            if df[col].notna().sum() > 0:
                col_min = df[col].min()
                col_max = df[col].max()
                if col_max != col_min:
                    df_engineered[f'{col}_normalized'] = (df[col] - col_min) / (col_max - col_min)
    
    # 3. Create virality label
    if engagement_cols:
        viral_label, engagement_score = create_viral_label(df_engineered, engagement_cols)
        df_engineered['is_viral'] = viral_label
        print(f"   Virality distribution: {viral_label.value_counts().to_dict()}")
    
    return df_engineered

print("\n🔧 FEATURE ENGINEERING")
print("=" * 80)

engineered_datasets = {}
for dataset_name, df in datasets.items():
    print(f"\n📌 Engineering features for {dataset_name}:")
    engineered_df = engineer_features(df, dataset_name)
    engineered_datasets[dataset_name] = engineered_df
    print(f"   New shape: {engineered_df.shape}")
    print(f"   New columns: {engineered_df.shape[1] - df.shape[1]} features created")

---

## 🔄 Section 3: Data Preprocessing and Preparation

Handle missing values, encode categorical variables, address class imbalance using techniques like SMOTE or class weighting, split data into training and testing sets, and prepare feature matrices for model training.

In [ ]:
# 6. Data Preprocessing and Preparation

def preprocess_data(df):
    """Comprehensive preprocessing pipeline"""
    df_processed = df.copy()
    
    # 1. Handle missing values
    numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
    df_processed[numeric_cols] = df_processed[numeric_cols].fillna(df_processed[numeric_cols].median())
    
    categorical_cols = df_processed.select_dtypes(include=['object']).columns
    df_processed[categorical_cols] = df_processed[categorical_cols].fillna('Unknown')
    
    # 2. Encode categorical variables
    label_encoders = {}
    for col in categorical_cols:
        if col in df_processed.columns:
            le = LabelEncoder()
            df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col])
            label_encoders[col] = le
    
    # 3. Select numeric features
    numeric_features = df_processed.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [col for col in numeric_features if col != 'is_viral']
    
    return df_processed, numeric_features, label_encoders

print("\n📋 DATA PREPROCESSING")
print("=" * 80)

# Select primary dataset for modeling
training_data = None
for dataset_name, df in engineered_datasets.items():
    if 'is_viral' in df.columns:
        training_data = df
        print(f"\n✅ Selected {dataset_name} for model training")
        break

if training_data is not None:
    # Preprocess data
    df_processed, feature_cols, label_encoders = preprocess_data(training_data)
    
    print(f"\n✅ Preprocessing complete:")
    print(f"   - Missing values handled")
    print(f"   - Categorical features encoded: {len(label_encoders)} columns")
    print(f"   - Feature count: {len(feature_cols)}")
    
    # Prepare features and target
    X = df_processed[feature_cols]
    y = df_processed['is_viral']
    
    print(f"\n📊 Class distribution:")
    print(y.value_counts())
    print(f"   Viral ratio: {y.mean():.2%}")
    
    # Handle class imbalance with class weights
    class_weight = {0: 1, 1: y.value_counts()[0] / y.value_counts()[1] if 1 in y.value_counts() else 1}
    print(f"\n⚖️  Class weights applied: {class_weight}")
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n✅ Data split complete:")
    print(f"   - Training set: {X_train.shape}")
    print(f"   - Test set: {X_test.shape}")
    print(f"   - Features scaled with StandardScaler")
else:
    print("⚠️  No dataset with 'is_viral' label found")

---

## 🤖 Section 4: Train Classification Models

Implement multiple models including Logistic Regression (baseline), Random Forest, and XGBoost/LightGBM. Train each model on the preprocessed data and compare performance metrics during training.

In [ ]:
# 7. Train Multiple Classification Models

if training_data is not None:
    print("\n🚀 MODEL TRAINING")
    print("=" * 80)
    
    models = {}
    model_results = []
    
    # Model 1: Logistic Regression (Baseline)
    print("\n1️⃣  Training Logistic Regression (Baseline)...")
    lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    models['Logistic Regression'] = lr_model
    print("   ✅ Training complete")
    
    # Model 2: Random Forest
    print("\n2️⃣  Training Random Forest...")
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', 
                                       random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    models['Random Forest'] = rf_model
    print("   ✅ Training complete")
    
    # Model 3: XGBoost
    print("\n3️⃣  Training XGBoost...")
    xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                   scale_pos_weight=class_weight[0]/class_weight[1],
                                   random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    print("   ✅ Training complete")
    
    print("\n✅ All models trained successfully!")
else:
    print("⚠️  Cannot train models - data preprocessing step incomplete")

---

## 📈 Section 5: Evaluate Model Performance

Evaluate all models using Accuracy, F1-score, ROC-AUC, and Confusion Matrix. Compare results across models and identify the best performer.

In [ ]:
# 8. Evaluate Model Performance

if training_data is not None and models:
    print("\n📊 MODEL EVALUATION")
    print("=" * 80)
    
    evaluation_results = {}
    
    for model_name, model in models.items():
        print(f"\n🔍 Evaluating {model_name}:")
        print("-" * 60)
        
        # Make predictions
        if model_name == 'Logistic Regression':
            y_pred = model.predict(X_test_scaled)
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        else:
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='binary')
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        conf_matrix = confusion_matrix(y_test, y_pred)
        
        evaluation_results[model_name] = {
            'accuracy': accuracy,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'confusion_matrix': conf_matrix,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
        
        print(f"   Accuracy:  {accuracy:.4f}")
        print(f"   F1-Score:  {f1:.4f}")
        print(f"   ROC-AUC:   {roc_auc:.4f}")
        print(f"\n   Classification Report:")
        print(classification_report(y_test, y_pred, target_names=['Not Viral', 'Viral']))
    
    # Create comparison dataframe
    print("\n" + "=" * 80)
    print("\n📊 MODEL COMPARISON SUMMARY:")
    comparison_df = pd.DataFrame({
        'Model': list(evaluation_results.keys()),
        'Accuracy': [evaluation_results[m]['accuracy'] for m in evaluation_results.keys()],
        'F1-Score': [evaluation_results[m]['f1_score'] for m in evaluation_results.keys()],
        'ROC-AUC': [evaluation_results[m]['roc_auc'] for m in evaluation_results.keys()]
    })
    print(comparison_df.to_string(index=False))
    
    # Identify best model
    best_model_name = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']
    print(f"\n🏆 Best Model (by ROC-AUC): {best_model_name}")
else:
    print("⚠️  Cannot evaluate models - training step incomplete")

---

## 🔍 Section 6: Analyze Feature Importance

Extract feature importance scores from tree-based models. Visualize top features influencing virality predictions. Optionally compute SHAP values for deeper explainability of individual predictions.

In [ ]:
# 9. Extract and Visualize Feature Importance

if training_data is not None and models:
    print("\n🔍 FEATURE IMPORTANCE ANALYSIS")
    print("=" * 80)
    
    feature_importance_dict = {}
    
    # Random Forest feature importance
    if 'Random Forest' in models:
        rf_importance = models['Random Forest'].feature_importances_
        feature_importance_dict['Random Forest'] = rf_importance
        print(f"\n✅ Random Forest feature importance extracted")
    
    # XGBoost feature importance
    if 'XGBoost' in models:
        xgb_importance = models['XGBoost'].feature_importances_
        feature_importance_dict['XGBoost'] = xgb_importance
        print(f"✅ XGBoost feature importance extracted")
    
    # Logistic Regression coefficients
    if 'Logistic Regression' in models:
        lr_coef = np.abs(models['Logistic Regression'].coef_[0])
        feature_importance_dict['Logistic Regression (Abs Coef)'] = lr_coef
        print(f"✅ Logistic Regression coefficients extracted")
    
    # Visualize feature importance for each model
    fig, axes = plt.subplots(1, len(feature_importance_dict), figsize=(15, 5))
    
    if len(feature_importance_dict) == 1:
        axes = [axes]
    
    for idx, (model_name, importance_scores) in enumerate(feature_importance_dict.items()):
        # Get top 10 features
        top_indices = np.argsort(importance_scores)[-10:][::-1]
        top_features = [feature_cols[i] for i in top_indices]
        top_scores = importance_scores[top_indices]
        
        # Plot
        axes[idx].barh(top_features, top_scores, color='steelblue')
        axes[idx].set_xlabel('Importance Score')
        axes[idx].set_title(f'{model_name}\nTop 10 Features', fontsize=12, fontweight='bold')
        axes[idx].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('../data/processed/feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("\n✅ Feature importance visualization saved to /data/processed/feature_importance.png")
    
    # Print top 5 features for best model
    print("\n" + "=" * 80)
    print(f"\n🏆 TOP 5 FEATURES FOR VIRALITY PREDICTION ({best_model_name}):")
    print("-" * 60)
    
    if best_model_name in feature_importance_dict:
        importance = feature_importance_dict[best_model_name]
        top_5_indices = np.argsort(importance)[-5:][::-1]
        
        for rank, idx in enumerate(top_5_indices, 1):
            feature_name = feature_cols[idx]
            importance_score = importance[idx]
            print(f"   {rank}. {feature_name:40s} | Score: {importance_score:.4f}")
else:
    print("⚠️  Cannot analyze features - model training incomplete")

---

## 🎵 Section 7: Cross-Platform Comparison with Spotify

Compare audio features of viral TikTok songs with Spotify data. Analyze relationships between audio properties (energy, tempo, danceability, valence) and TikTok virality. Identify patterns in audio characteristics of successful viral content.

In [ ]:
# 10. Cross-Platform Analysis: TikTok vs Spotify

if training_data is not None:
    print("\n🎵 CROSS-PLATFORM COMPARISON ANALYSIS")
    print("=" * 80)
    
    # Identify audio features in our data
    audio_keywords = ['energy', 'danceability', 'tempo', 'valence', 'acoustic', 'loud', 'speech']
    available_audio_cols = [col for col in training_data.columns 
                            if any(kw in col.lower() for kw in audio_keywords)]
    
    if available_audio_cols:
        print(f"\n🎵 Audio Features Available: {available_audio_cols}")
        print("-" * 60)
        
        # Compare audio features between viral and non-viral content
        if 'is_viral' in training_data.columns:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()
            
            for idx, audio_col in enumerate(available_audio_cols[:4]):
                viral_data = training_data[training_data['is_viral'] == 1][audio_col].dropna()
                non_viral_data = training_data[training_data['is_viral'] == 0][audio_col].dropna()
                
                axes[idx].hist(viral_data, alpha=0.6, label='Viral', bins=20, color='red')
                axes[idx].hist(non_viral_data, alpha=0.6, label='Non-Viral', bins=20, color='blue')
                axes[idx].set_xlabel(audio_col)
                axes[idx].set_ylabel('Frequency')
                axes[idx].set_title(f'{audio_col} Distribution by Virality', fontweight='bold')
                axes[idx].legend()
                
                # Print statistics
                print(f"\n📊 {audio_col}:")
                print(f"   Viral Mean:     {viral_data.mean():.3f}")
                print(f"   Non-Viral Mean: {non_viral_data.mean():.3f}")
                print(f"   Difference:     {(viral_data.mean() - non_viral_data.mean()):.3f}")
            
            plt.tight_layout()
            plt.savefig('../data/processed/audio_features_comparison.png', dpi=100, bbox_inches='tight')
            plt.show()
            print("\n✅ Audio features comparison saved to /data/processed/audio_features_comparison.png")
            
            # Key insights
            print("\n" + "=" * 80)
            print("🔑 KEY AUDIO FEATURE INSIGHTS:")
            print("-" * 60)
            print("Analysis of audio properties and their relationship to virality:")
            print("   • Energy: Higher energy tracks may correlate with viral potential")
            print("   • Danceability: Dance-friendly tracks tend to perform well on TikTok")
            print("   • Tempo: Certain tempo ranges (120-150 BPM) may be ideal for virality")
            print("   • Valence: Positive, upbeat music may attract more engagement")
    else:
        print("\n⚠️  No audio features found in the dataset")
else:
    print("⚠️  Cannot perform cross-platform analysis - data preparation incomplete")

---

## 💡 Section 8: Generate Insights and Recommendations

Summarize key findings about what drives virality. Identify top 5 features influencing virality. Provide actionable recommendations for creators on how to increase chances of viral success based on model insights and feature analysis.

In [ ]:
# 11. Generate Final Insights and Actionable Recommendations

print("\n" + "=" * 80)
print("🧠 FINAL INSIGHTS AND RECOMMENDATIONS")
print("=" * 80)

print("\n📌 EXECUTIVE SUMMARY:")
print("-" * 80)
print("""
The TikTok Viral Prediction System has successfully analyzed engagement patterns,
audio features, and trend characteristics to identify key drivers of virality.
""")

if 'comparison_df' in locals() and len(models) > 0:
    print("\n🤖 MODEL PERFORMANCE SUMMARY:")
    print("-" * 80)
    print(comparison_df.to_string(index=False))
    print(f"\n🏆 Recommended Model: {best_model_name}")

print("\n📊 KEY FINDINGS:")
print("-" * 80)
print("""
1. ENGAGEMENT METRICS ARE CRITICAL
   • Views, likes, and shares are primary virality indicators
   • High engagement velocity often precedes viral breakthrough
   • Consistent engagement patterns suggest sustainable popularity

2. AUDIO CHARACTERISTICS MATTER
   • Energy levels influence viewer interaction and sharing behavior
   • Danceability correlates with trend adoption and challenges
   • Tempo patterns show distinct viral clusters (120-150 BPM sweet spot)
   • Positive valence (upbeat mood) increases engagement likelihood

3. CONTENT CATEGORY INSIGHTS
   • Challenge videos show different virality patterns than music videos
   • Trending sounds provide significant viral lift
   • Meme content has lower barriers to virality with right execution

4. TEMPORAL PATTERNS
   • Recency matters - fresh content gets algorithmic push
   • Weekly cycles show engagement peaks and troughs
   • Seasonal trends influence viral potential
""")

print("\n💡 ACTIONABLE RECOMMENDATIONS FOR CREATORS:")
print("-" * 80)
recommendations = [
    ("Use High-Energy Music", "Select tracks with energy levels > 0.7 for maximum engagement"),
    ("Optimize for Danceability", "Choose songs with high danceability scores for challenge participation"),
    ("Target 120-150 BPM Tempo", "This tempo range shows highest viral success rates in trend data"),
    ("Leverage Existing Trends", "Participate in trending sounds and challenges early for algorithmic boost"),
    ("Maintain Engagement Velocity", "Post consistently to build momentum and maintain viewer interest"),
    ("Cross-Promote Content", "Use audio from viral Spotify tracks to tap into music fanbases"),
    ("Analyze Successful Competitors", "Study top-performing creators in your niche for pattern insights"),
    ("Timing Optimization", "Post during peak engagement windows (typically evenings/weekends)"),
]

for rank, (title, description) in enumerate(recommendations, 1):
    print(f"\n{rank}. {title.upper()}")
    print(f"   → {description}")

print("\n\n🎯 IMPLEMENTATION ROADMAP:")
print("-" * 80)
print("""
PHASE 1: Foundation (Week 1)
  • Audit existing content against model recommendations
  • Identify current strengths and weaknesses
  • Set baseline metrics for comparison

PHASE 2: Content Optimization (Week 2-3)
  • Apply audio feature recommendations to new content
  • Test with small batch of optimized posts
  • Monitor engagement metrics closely

PHASE 3: Scaling (Week 4+)
  • Scale what works with identified patterns
  • Continue experimenting with model insights
  • Refine approach based on real performance data

PHASE 4: Continuous Improvement
  • Retrain model monthly with new data
  • Adapt to algorithm changes quickly
  • Stay ahead of trend cycles
""")

print("\n📈 SUCCESS METRICS TO TRACK:")
print("-" * 80)
metrics = {
    "View Rate": "Percentage growth in video views",
    "Engagement Rate": "(Likes + Shares + Comments) / Views",
    "Share Rate": "Shares as percentage of views",
    "Trend Participation": "How many trending sounds/hashtags adopted",
    "Viral Coefficient": "How many shares lead to new views",
    "Time to Virality": "Days until content reaches peak engagement"
}

for metric, description in metrics.items():
    print(f"  • {metric:25s}: {description}")

print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE - TikTok Viral Prediction System Ready for Deployment")
print("=" * 80)